# Análise Exploratória — Bronze

Este notebook valida os dados que chegaram na camada Bronze após a execução do `etl_bronze.py`.

O objetivo é responder as seguintes perguntas:
- O volume de dados está correto comparado à fonte?
- As colunas foram padronizadas para lowercase?
- Os nulos foram preservados?
- As partições estão corretas?
- O que precisa ser tratado no Silver?

As conclusões documentadas aqui embasam as decisões de transformação no `etl_silver.py`.

**Database:** `ifood_case_bronze`  
**Tabelas:** `table_yellow_taxi_bronze` e `table_green_taxi_bronze`

## 1. Importações e configurações

In [10]:
import logging
import awswrangler as wr
import pandas as pd
import warnings

# ─── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] exploratory_bronze - %(message)s",
    datefmt="%d-%m-%Y %H:%M:%S",
)
logger = logging.getLogger("exploratory_bronze")

# Silencia logs desnecessários do boto3 e awswrangler
logging.getLogger("boto3").setLevel(logging.WARNING)
logging.getLogger("botocore").setLevel(logging.WARNING)
logging.getLogger("awswrangler").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=UserWarning, module="awswrangler")

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATABASE  = "ifood_case_bronze"
TABELAS   = ["table_yellow_taxi_bronze", "table_green_taxi_bronze"]
MESES     = [1, 2, 3, 4, 5]
S3_OUTPUT = "s3://ifood-case-data-lake-kaique/athena-results/"

logger.info("Bibliotecas carregadas com sucesso")

07-06-2026 18:05:22 [INFO] exploratory_bronze - Bibliotecas carregadas com sucesso


## 2. Verificação de disponibilidade das tabelas

Verificamos se as tabelas foram criadas corretamente no Glue Catalog
após a execução do Crawler Bronze.

In [11]:
logger.info("Verificando tabelas no Glue Catalog | database=%s", DATABASE)

tabelas_catalog = wr.catalog.get_tables(database=DATABASE)
df_catalog = pd.DataFrame(tabelas_catalog)[["Name", "DatabaseName", "TableType"]]

logger.info("Tabelas encontradas | total=%s", len(df_catalog))

display(
    df_catalog.style
    .set_caption("Tabelas no Glue Catalog — ifood_case_bronze")
    .set_properties(**{"text-align": "left"})
)

07-06-2026 18:05:25 [INFO] exploratory_bronze - Verificando tabelas no Glue Catalog | database=ifood_case_bronze
07-06-2026 18:05:26 [INFO] exploratory_bronze - Tabelas encontradas | total=2


,Name,DatabaseName,TableType
0,table_green_taxi_bronze,ifood_case_bronze,EXTERNAL_TABLE
1,table_yellow_taxi_bronze,ifood_case_bronze,EXTERNAL_TABLE


## 3. Volume de dados por tabela e mês

Comparamos o volume de registros no Bronze com o que foi identificado
na análise exploratória da fonte (`exploratory_source.ipynb`).

In [12]:
logger.info("Iniciando analise de volume por tabela e mes")

volumes = []

for tabela in TABELAS:
    for mes in MESES:
        df = wr.athena.read_sql_query(
            sql=f"""
                SELECT COUNT(*) as total_linhas
                FROM {tabela}
                WHERE partition_year = '2023'
                AND   partition_month = '{mes}'
            """,
            database=DATABASE,
            s3_output=S3_OUTPUT
        )
        total = df["total_linhas"].iloc[0]
        volumes.append({
            "tabela": tabela,
            "mes": f"2023-{str(mes).zfill(2)}",
            "linhas": total,
        })
        logger.info("Volume | tabela=%s | mes=%s | linhas=%s", tabela, mes, f"{total:,}")

df_volumes = pd.DataFrame(volumes)

logger.info("Analise de volume concluida")

display(
    df_volumes
    .set_index(["tabela", "mes"])
    .rename(columns={"linhas": "Linhas"})
    .style
    .set_caption("Volume por Tabela e Mes — Bronze")
    .format("{:,}", subset=["Linhas"])
    .set_properties(**{"text-align": "right"})
)

07-06-2026 18:05:33 [INFO] exploratory_bronze - Iniciando analise de volume por tabela e mes
07-06-2026 18:05:43 [INFO] exploratory_bronze - Volume | tabela=table_yellow_taxi_bronze | mes=1 | linhas=3,066,766
07-06-2026 18:05:51 [INFO] exploratory_bronze - Volume | tabela=table_yellow_taxi_bronze | mes=2 | linhas=2,913,955
07-06-2026 18:06:01 [INFO] exploratory_bronze - Volume | tabela=table_yellow_taxi_bronze | mes=3 | linhas=3,403,766
07-06-2026 18:06:11 [INFO] exploratory_bronze - Volume | tabela=table_yellow_taxi_bronze | mes=4 | linhas=3,288,250
07-06-2026 18:06:21 [INFO] exploratory_bronze - Volume | tabela=table_yellow_taxi_bronze | mes=5 | linhas=3,513,649
07-06-2026 18:06:31 [INFO] exploratory_bronze - Volume | tabela=table_green_taxi_bronze | mes=1 | linhas=68,211
07-06-2026 18:06:39 [INFO] exploratory_bronze - Volume | tabela=table_green_taxi_bronze | mes=2 | linhas=64,809
07-06-2026 18:06:47 [INFO] exploratory_bronze - Volume | tabela=table_green_taxi_bronze | mes=3 | linha

## 4. Validação do lowercase das colunas

Verificamos se todas as colunas foram padronizadas para lowercase
conforme aplicado no `transform` do `etl_bronze.py`.

In [13]:
logger.info("Validando padronizacao lowercase das colunas")

for tabela in TABELAS:
    colunas = wr.catalog.get_table_types(database=DATABASE, table=tabela)
    colunas_upper = [col for col in colunas.keys() if col != col.lower()]

    if colunas_upper:
        logger.warning("Colunas nao lowercase | tabela=%s | colunas=%s", tabela, colunas_upper)
    else:
        logger.info("Todas as colunas em lowercase | tabela=%s", tabela)

    df_schema = pd.DataFrame({
        "coluna": list(colunas.keys()),
        "tipo": list(colunas.values()),
        "lowercase": ["✓" if col == col.lower() else "✗" for col in colunas.keys()],
    })

    display(
        df_schema
        .set_index("coluna")
        .style
        .set_caption(f"Schema — {tabela}")
        .apply(lambda col: [
            "background-color: #ffcccc; color: #842029; font-weight: bold" if v == "✗"
            else "background-color: #d1e7dd; color: #0a3622; font-weight: bold"
            for v in col
        ], subset=["lowercase"])
        .set_properties(**{"text-align": "left"})
    )

07-06-2026 18:07:08 [INFO] exploratory_bronze - Validando padronizacao lowercase das colunas
07-06-2026 18:07:09 [INFO] exploratory_bronze - Todas as colunas em lowercase | tabela=table_yellow_taxi_bronze


,tipo,lowercase
coluna,,
vendorid,bigint,✓
tpep_pickup_datetime,timestamp,✓
tpep_dropoff_datetime,timestamp,✓
passenger_count,double,✓
trip_distance,double,✓
ratecodeid,double,✓
store_and_fwd_flag,string,✓
pulocationid,bigint,✓
dolocationid,bigint,✓


07-06-2026 18:07:10 [INFO] exploratory_bronze - Todas as colunas em lowercase | tabela=table_green_taxi_bronze


,tipo,lowercase
coluna,,
vendorid,bigint,✓
lpep_pickup_datetime,timestamp,✓
lpep_dropoff_datetime,timestamp,✓
store_and_fwd_flag,string,✓
ratecodeid,double,✓
pulocationid,bigint,✓
dolocationid,bigint,✓
passenger_count,double,✓
trip_distance,double,✓


## 5. Amostra dos dados

Visualizamos uma amostra dos dados para validar que o conteúdo
chegou corretamente no Bronze conforme esperado da fonte.

In [14]:
logger.info("Carregando amostra dos dados")

for tabela in TABELAS:
    df_sample = wr.athena.read_sql_query(
        sql=f"""
            SELECT *
            FROM {tabela}
            WHERE partition_year = '2023'
            AND   partition_month = '1'
            LIMIT 5
        """,
        database=DATABASE,
        s3_output=S3_OUTPUT,
    )
    logger.info("Amostra carregada | tabela=%s | linhas=%s", tabela, len(df_sample))

    display(
        df_sample.style
        .set_caption(f"Amostra — {tabela} (Janeiro 2023)")
        .set_properties(**{"text-align": "left"})
    )

07-06-2026 18:07:10 [INFO] exploratory_bronze - Carregando amostra dos dados
07-06-2026 18:07:20 [INFO] exploratory_bronze - Amostra carregada | tabela=table_yellow_taxi_bronze | linhas=5


,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,partition_year,partition_month
0,2,2023-01-22 21:37:32,2023-01-22 21:53:13,3.000000,3.120000,1.000000,N,230,211,1,17.000000,1.000000,0.500000,4.400000,0.000000,1.000000,26.400000,2.500000,0.000000,2023,1
1,2,2023-01-22 21:56:16,2023-01-22 22:05:41,1.000000,1.940000,1.000000,N,114,13,1,12.100000,1.000000,0.500000,3.420000,0.000000,1.000000,20.520000,2.500000,0.000000,2023,1
2,2,2023-01-22 21:39:05,2023-01-22 21:43:20,1.000000,0.370000,1.000000,N,229,229,1,5.800000,1.000000,0.500000,0.250000,0.000000,1.000000,11.050000,2.500000,0.000000,2023,1
3,2,2023-01-22 21:50:14,2023-01-22 21:52:57,1.000000,0.850000,1.000000,N,262,263,1,5.800000,1.000000,0.500000,2.160000,0.000000,1.000000,12.960000,2.500000,0.000000,2023,1
4,2,2023-01-22 21:57:31,2023-01-22 22:02:46,1.000000,1.230000,1.000000,N,141,75,2,7.900000,1.000000,0.500000,0.000000,0.000000,1.000000,12.900000,2.500000,0.000000,2023,1


07-06-2026 18:07:28 [INFO] exploratory_bronze - Amostra carregada | tabela=table_green_taxi_bronze | linhas=5


,vendorid,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,ratecodeid,pulocationid,dolocationid,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,partition_year,partition_month
0,2,2023-01-01 00:26:10,2023-01-01 00:37:11,N,1.000000,166,143,1.000000,2.580000,14.900000,1.000000,0.500000,4.030000,0.000000,nan,1.000000,24.180000,1.000000,1.000000,2.750000,2023,1
1,2,2023-01-01 00:51:03,2023-01-01 00:57:49,N,1.000000,24,43,1.000000,1.810000,10.700000,1.000000,0.500000,2.640000,0.000000,nan,1.000000,15.840000,1.000000,1.000000,0.000000,2023,1
2,2,2023-01-01 00:35:12,2023-01-01 00:41:32,N,1.000000,223,179,1.000000,0.000000,7.200000,1.000000,0.500000,1.940000,0.000000,nan,1.000000,11.640000,1.000000,1.000000,0.000000,2023,1
3,1,2023-01-01 00:13:14,2023-01-01 00:19:03,N,1.000000,41,238,1.000000,1.300000,6.500000,0.500000,1.500000,1.700000,0.000000,nan,1.000000,10.200000,1.000000,1.000000,0.000000,2023,1
4,1,2023-01-01 00:33:04,2023-01-01 00:39:02,N,1.000000,41,74,1.000000,1.100000,6.000000,0.500000,1.500000,0.000000,0.000000,nan,1.000000,8.000000,1.000000,1.000000,0.000000,2023,1


## 6. Validação de nulos nas colunas obrigatórias

Verificamos se os nulos foram preservados no Bronze
conforme esperado — o Bronze não deve remover nulos.

In [15]:
logger.info("Validando nulos nas colunas obrigatorias")

colunas_obrigatorias = {
    "table_yellow_taxi_bronze": [
        "vendorid", "passenger_count", "total_amount",
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
    ],
    "table_green_taxi_bronze": [
        "vendorid", "passenger_count", "total_amount",
        "lpep_pickup_datetime", "lpep_dropoff_datetime",
    ],
}

for tabela, colunas in colunas_obrigatorias.items():
    rows = []
    for col in colunas:
        df = wr.athena.read_sql_query(
            sql=f"""
                SELECT
                    COUNT(*) as total,
                    COUNT({col}) as nao_nulos,
                    COUNT(*) - COUNT({col}) as nulos
                FROM {tabela}
            """,
            database=DATABASE,
        )
        total = df["total"].iloc[0]
        nulos = df["nulos"].iloc[0]
        pct   = round(nulos / total * 100, 2) if total > 0 else 0
        rows.append({
            "coluna": col,
            "total": total,
            "nulos": nulos,
            "pct_nulos": pct,
        })
        logger.info(
            "Nulos | tabela=%s | coluna=%s | nulos=%s | pct=%s%%",
            tabela, col, f"{nulos:,}", pct
        )

    df_nulos = pd.DataFrame(rows)

    display(
        df_nulos
        .set_index("coluna")
        .style
        .set_caption(f"Nulos nas Colunas Obrigatorias — {tabela}")
        .format("{:,}", subset=["total", "nulos"])
        .format("{:.2f}%", subset=["pct_nulos"])
        .bar(subset=["pct_nulos"], color="#ffcccc", vmin=0, vmax=100)
        .set_properties(**{"text-align": "left"})
    )

07-06-2026 18:07:28 [INFO] exploratory_bronze - Validando nulos nas colunas obrigatorias
07-06-2026 18:07:42 [INFO] exploratory_bronze - Nulos | tabela=table_yellow_taxi_bronze | coluna=vendorid | nulos=0 | pct=0.0%
07-06-2026 18:07:56 [INFO] exploratory_bronze - Nulos | tabela=table_yellow_taxi_bronze | coluna=passenger_count | nulos=428,665 | pct=2.65%
07-06-2026 18:08:09 [INFO] exploratory_bronze - Nulos | tabela=table_yellow_taxi_bronze | coluna=total_amount | nulos=0 | pct=0.0%
07-06-2026 18:08:22 [INFO] exploratory_bronze - Nulos | tabela=table_yellow_taxi_bronze | coluna=tpep_pickup_datetime | nulos=0 | pct=0.0%
07-06-2026 18:08:36 [INFO] exploratory_bronze - Nulos | tabela=table_yellow_taxi_bronze | coluna=tpep_dropoff_datetime | nulos=0 | pct=0.0%


,total,nulos,pct_nulos
coluna,,,
vendorid,"16,186,386",0,0.00%
passenger_count,"16,186,386","428,665",2.65%
total_amount,"16,186,386",0,0.00%
tpep_pickup_datetime,"16,186,386",0,0.00%
tpep_dropoff_datetime,"16,186,386",0,0.00%


07-06-2026 18:08:49 [INFO] exploratory_bronze - Nulos | tabela=table_green_taxi_bronze | coluna=vendorid | nulos=0 | pct=0.0%
07-06-2026 18:09:01 [INFO] exploratory_bronze - Nulos | tabela=table_green_taxi_bronze | coluna=passenger_count | nulos=22,896 | pct=6.74%
07-06-2026 18:09:14 [INFO] exploratory_bronze - Nulos | tabela=table_green_taxi_bronze | coluna=total_amount | nulos=0 | pct=0.0%
07-06-2026 18:09:30 [INFO] exploratory_bronze - Nulos | tabela=table_green_taxi_bronze | coluna=lpep_pickup_datetime | nulos=0 | pct=0.0%
07-06-2026 18:09:41 [INFO] exploratory_bronze - Nulos | tabela=table_green_taxi_bronze | coluna=lpep_dropoff_datetime | nulos=0 | pct=0.0%


,total,nulos,pct_nulos
coluna,,,
vendorid,"339,630",0,0.00%
passenger_count,"339,630","22,896",6.74%
total_amount,"339,630",0,0.00%
lpep_pickup_datetime,"339,630",0,0.00%
lpep_dropoff_datetime,"339,630",0,0.00%


## 7. Validação do particionamento

Verificamos se as partições foram criadas corretamente no S3
com a estrutura `partition_year` e `partition_month`.

In [16]:
logger.info("Validando particionamento das tabelas")

for tabela in TABELAS:
    df = wr.athena.read_sql_query(
        sql=f"""
            SELECT
                partition_year,
                partition_month,
                COUNT(*) as total_linhas
            FROM {tabela}
            GROUP BY partition_year, partition_month
            ORDER BY partition_year, partition_month
        """,
        database=DATABASE,
        s3_output=S3_OUTPUT,
    )
    logger.info("Particoes encontradas | tabela=%s | total=%s", tabela, len(df))

    display(
        df.style
        .set_caption(f"Particionamento — {tabela}")
        .format("{:,}", subset=["total_linhas"])
        .set_properties(**{"text-align": "right"})
    )

07-06-2026 18:09:41 [INFO] exploratory_bronze - Validando particionamento das tabelas
07-06-2026 18:09:50 [INFO] exploratory_bronze - Particoes encontradas | tabela=table_yellow_taxi_bronze | total=5


,partition_year,partition_month,total_linhas
0,2023,1,"3,066,766"
1,2023,2,"2,913,955"
2,2023,3,"3,403,766"
3,2023,4,"3,288,250"
4,2023,5,"3,513,649"


07-06-2026 18:09:57 [INFO] exploratory_bronze - Particoes encontradas | tabela=table_green_taxi_bronze | total=5


,partition_year,partition_month,total_linhas
0,2023,1,"68,211"
1,2023,2,"64,809"
2,2023,3,"72,044"
3,2023,4,"65,392"
4,2023,5,"69,174"


## 8. Conclusões

Com base na validação da camada Bronze, as seguintes decisões foram tomadas para o `etl_silver.py`:

| Observação | Decisão |
|---|---|
| Volume consistente com a fonte | Bronze ingerido corretamente |
| Todas as colunas em lowercase | Padronização aplicada com sucesso no Bronze |
| Nulos preservados nas colunas obrigatórias | Remover nulos no Silver |
| `passenger_count` com nulos e tipo `double` | Converter para `int` e remover nulos no Silver |
| `tpep_*` no Yellow e `lpep_*` no Green | Renomear para `pickup_datetime` e `dropoff_datetime` no Silver |
| `partition_year` e `partition_month` como `string` | Partições do Bronze são string por padrão do Crawler — no Silver definir como `int` via Terraform |